# Ch.6 — Infrastructure as Code (Azure Integration)

> **Goal:** Provision Azure resources with Terraform — Container Instances, Storage Accounts, and Resource Groups. Learn remote state management with Azure backends.
>
> **Tech stack:** Terraform, Azure Provider, Azure CLI
>
> **Prerequisites:**
> - Azure subscription (free tier available)
> - Azure CLI installed
> - Terraform installed (from main notebook)
>
> **What you'll build:** Production-ready Terraform configurations for Azure cloud resources with remote state backends.

In [ ]:
# Cell 1: Azure credentials setup

# IMPORTANT: These credentials will be stripped by the pre-push hook
# Never commit real credentials to Git

# Azure subscription details
AZURE_SUBSCRIPTION_ID = ""  # Your subscription ID
AZURE_TENANT_ID = ""        # Your tenant ID
AZURE_CLIENT_ID = ""        # Service principal client ID  
AZURE_CLIENT_SECRET = ""    # Service principal client secret

# Resource naming
AZURE_RESOURCE_GROUP = "terraform-demo-rg"
AZURE_LOCATION = "eastus"
AZURE_STORAGE_ACCOUNT = "tfstate" + "".join([str(i) for i in range(5)])  # Must be globally unique

print("📋 Azure Configuration:")
print(f"   Subscription ID: {AZURE_SUBSCRIPTION_ID[:8]}..." if AZURE_SUBSCRIPTION_ID else "   ⚠️ Subscription ID not set")
print(f"   Resource Group: {AZURE_RESOURCE_GROUP}")
print(f"   Location: {AZURE_LOCATION}")
print(f"   Storage Account: {AZURE_STORAGE_ACCOUNT}")

# Authenticate with Azure CLI
print("\n🔐 Authenticating with Azure...")
print("   Run: az login")
print("   Then: az account set --subscription <YOUR_SUBSCRIPTION_ID>")

In [ ]:
# Cell 2: Terraform Azure provider setup

import os

# Create Azure project directory
azure_dir = "terraform_azure_demo"
os.makedirs(azure_dir, exist_ok=True)

# Create provider configuration
provider_tf = f"""terraform {{
  required_providers {{
    azurerm = {{
      source  = "hashicorp/azurerm"
      version = "~> 3.0"
    }}
  }}
  required_version = ">= 1.5"
}}

provider "azurerm" {{
  features {{
    resource_group {{
      prevent_deletion_if_contains_resources = false
    }}
  }}
  
  # Authentication via Azure CLI (already logged in)
  # Alternative: use environment variables or service principal
  subscription_id = var.subscription_id
}}
"""

with open(f"{azure_dir}/provider.tf", "w") as f:
    f.write(provider_tf)

# Create variables
variables_tf = f"""variable "subscription_id" {{
  description = "Azure subscription ID"
  type        = string
  default     = "{AZURE_SUBSCRIPTION_ID}"
  sensitive   = true
}}

variable "resource_group_name" {{
  description = "Name of the resource group"
  type        = string
  default     = "{AZURE_RESOURCE_GROUP}"
}}

variable "location" {{
  description = "Azure region"
  type        = string
  default     = "{AZURE_LOCATION}"
}}

variable "storage_account_name" {{
  description = "Storage account name (globally unique)"
  type        = string
  default     = "{AZURE_STORAGE_ACCOUNT}"
}}

variable "environment" {{
  description = "Environment tag"
  type        = string
  default     = "dev"
}}
"""

with open(f"{azure_dir}/variables.tf", "w") as f:
    f.write(variables_tf)

print("✅ Azure provider configuration created:")
print(f"   - {azure_dir}/provider.tf")
print(f"   - {azure_dir}/variables.tf")

In [ ]:
# Cell 3: Provision Azure Container Instance with Terraform

# Create main.tf for Container Instance
main_tf = """# Resource Group
resource "azurerm_resource_group" "main" {
  name     = var.resource_group_name
  location = var.location
  
  tags = {
    Environment = var.environment
    ManagedBy   = "Terraform"
    Purpose     = "Demo"
  }
}

# Azure Container Instance (ACI)
resource "azurerm_container_group" "nginx" {
  name                = "nginx-terraform"
  location            = azurerm_resource_group.main.location
  resource_group_name = azurerm_resource_group.main.name
  os_type             = "Linux"
  dns_name_label      = "nginx-terraform-${var.environment}"
  
  container {
    name   = "nginx"
    image  = "nginx:latest"
    cpu    = "0.5"
    memory = "1.5"
    
    ports {
      port     = 80
      protocol = "TCP"
    }
    
    environment_variables = {
      "ENVIRONMENT" = var.environment
    }
  }
  
  tags = {
    Environment = var.environment
    ManagedBy   = "Terraform"
  }
}
"""

with open(f"{azure_dir}/main.tf", "w") as f:
    f.write(main_tf)

# Create outputs
outputs_tf = """output "resource_group_name" {
  description = "Name of the resource group"
  value       = azurerm_resource_group.main.name
}

output "container_fqdn" {
  description = "Fully qualified domain name of the container"
  value       = azurerm_container_group.nginx.fqdn
}

output "container_url" {
  description = "URL to access the container"
  value       = "http://${azurerm_container_group.nginx.fqdn}"
}

output "container_ip" {
  description = "IP address of the container"
  value       = azurerm_container_group.nginx.ip_address
}
"""

with open(f"{azure_dir}/outputs.tf", "w") as f:
    f.write(outputs_tf)

print("✅ Azure Container Instance configuration created:")
print(f"   - {azure_dir}/main.tf")
print(f"   - {azure_dir}/outputs.tf")
print("\n📋 To provision:")
print("   cd terraform_azure_demo")
print("   terraform init")
print("   terraform plan")
print("   terraform apply")
print("\n   The container will be accessible at: http://nginx-terraform-dev.<region>.azurecontainer.io")

In [ ]:
# Cell 4: Provision Azure Storage Account

# Add storage account to main.tf
storage_config = """
# Azure Storage Account
resource "azurerm_storage_account" "main" {
  name                     = var.storage_account_name
  resource_group_name      = azurerm_resource_group.main.name
  location                 = azurerm_resource_group.main.location
  account_tier             = "Standard"
  account_replication_type = "LRS"  # Locally redundant storage
  
  # Enable blob versioning
  blob_properties {
    versioning_enabled = true
    
    delete_retention_policy {
      days = 7
    }
  }
  
  tags = {
    Environment = var.environment
    ManagedBy   = "Terraform"
  }
}

# Storage Container (blob container)
resource "azurerm_storage_container" "data" {
  name                  = "data"
  storage_account_name  = azurerm_storage_account.main.name
  container_access_type = "private"
}

# Storage Container for backups
resource "azurerm_storage_container" "backups" {
  name                  = "backups"
  storage_account_name  = azurerm_storage_account.main.name
  container_access_type = "private"
}
"""

# Read existing main.tf
with open(f"{azure_dir}/main.tf", "r") as f:
    existing_content = f.read()

# Append storage config
with open(f"{azure_dir}/main.tf", "w") as f:
    f.write(existing_content + "\n" + storage_config)

# Add storage outputs
storage_outputs = """
output "storage_account_name" {
  description = "Name of the storage account"
  value       = azurerm_storage_account.main.name
}

output "storage_account_primary_key" {
  description = "Primary access key"
  value       = azurerm_storage_account.main.primary_access_key
  sensitive   = true
}

output "storage_account_endpoint" {
  description = "Blob storage endpoint"
  value       = azurerm_storage_account.main.primary_blob_endpoint
}
"""

with open(f"{azure_dir}/outputs.tf", "a") as f:
    f.write(storage_outputs)

print("✅ Azure Storage Account configuration added:")
print("   - Storage account with LRS replication")
print("   - Two containers: 'data' and 'backups'")
print("   - Blob versioning enabled")
print("   - 7-day delete retention policy")
print("\n💡 Storage account names must be:")
print("   - 3-24 characters")
print("   - Lowercase letters and numbers only")
print("   - Globally unique across all Azure")

In [ ]:
# Cell 5: Manage Azure resource groups

# Create resource group management utilities
rg_management = """# Additional resource groups for different environments

resource "azurerm_resource_group" "staging" {
  name     = "${var.resource_group_name}-staging"
  location = var.location
  
  tags = {
    Environment = "staging"
    ManagedBy   = "Terraform"
  }
}

resource "azurerm_resource_group" "prod" {
  name     = "${var.resource_group_name}-prod"
  location = "westus2"  # Different region for prod
  
  tags = {
    Environment = "prod"
    ManagedBy   = "Terraform"
  }
  
  # Prevent accidental deletion of production resources
  lifecycle {
    prevent_destroy = true
  }
}

# Lock production resource group (requires manual unlock to delete)
resource "azurerm_management_lock" "prod_lock" {
  name       = "prod-lock"
  scope      = azurerm_resource_group.prod.id
  lock_level = "CanNotDelete"
  notes      = "Production resources - requires manual unlock to delete"
}
"""

with open(f"{azure_dir}/resource_groups.tf", "w") as f:
    f.write(rg_management)

print("✅ Resource group management created:")
print("   - dev: {AZURE_LOCATION}")
print("   - staging: {AZURE_LOCATION}")
print("   - prod: westus2 (different region)")
print("\n🔒 Production safeguards:")
print("   - prevent_destroy lifecycle rule")
print("   - CanNotDelete management lock")
print("   - Requires manual unlock: az lock delete --name prod-lock --resource-group <name>")
print("\n💡 Resource group best practices:")
print("   - Group resources by environment and lifecycle")
print("   - Use tags for cost tracking and automation")
print("   - Enable resource locks for critical resources")
print("   - Set up budgets and alerts per resource group")

In [ ]:
# Cell 6: Remote state backend (Azure Storage)

# Create backend configuration
backend_config = """# Remote State Backend Configuration
# 
# Store Terraform state in Azure Blob Storage instead of local file
# Benefits:
#   - Team collaboration (shared state)
#   - State locking (prevents concurrent modifications)
#   - Encryption at rest
#   - Version history
#   - Automated backups

terraform {
  backend "azurerm" {
    resource_group_name  = "terraform-state-rg"
    storage_account_name = "tfstateXXXXX"  # Replace with your storage account
    container_name       = "tfstate"
    key                  = "infrastructure.tfstate"
    
    # Optional: use service principal for authentication
    # client_id            = var.client_id
    # client_secret        = var.client_secret
    # tenant_id            = var.tenant_id
    # subscription_id      = var.subscription_id
  }
}
"""

# Create a separate file for backend
with open(f"{azure_dir}/backend.tf.example", "w") as f:
    f.write(backend_config)

# Create script to set up state backend
setup_script = f"""#!/bin/bash
# Setup script for Azure remote state backend

# Variables
RESOURCE_GROUP="terraform-state-rg"
STORAGE_ACCOUNT="tfstate$(openssl rand -hex 4)"
CONTAINER_NAME="tfstate"
LOCATION="{AZURE_LOCATION}"

echo "🔧 Setting up Terraform remote state backend..."

# Create resource group
az group create --name $RESOURCE_GROUP --location $LOCATION

# Create storage account
az storage account create \\
  --name $STORAGE_ACCOUNT \\
  --resource-group $RESOURCE_GROUP \\
  --location $LOCATION \\
  --sku Standard_LRS \\
  --encryption-services blob

# Get storage account key
ACCOUNT_KEY=$(az storage account keys list \\
  --resource-group $RESOURCE_GROUP \\
  --account-name $STORAGE_ACCOUNT \\
  --query '[0].value' -o tsv)

# Create blob container
az storage container create \\
  --name $CONTAINER_NAME \\
  --account-name $STORAGE_ACCOUNT \\
  --account-key $ACCOUNT_KEY

echo "✅ Remote state backend created!"
echo ""
echo "📋 Update backend.tf with these values:"
echo "   resource_group_name  = \\"$RESOURCE_GROUP\\""
echo "   storage_account_name = \\"$STORAGE_ACCOUNT\\""
echo "   container_name       = \\"$CONTAINER_NAME\\""
echo ""
echo "🔐 Initialize Terraform with remote backend:"
echo "   terraform init -backend-config=\\"backend.tf\\""
"""

with open(f"{azure_dir}/setup_backend.sh", "w") as f:
    f.write(setup_script)

# Create PowerShell version
setup_ps1 = f"""# Setup script for Azure remote state backend (PowerShell)

$ResourceGroup = "terraform-state-rg"
$StorageAccount = "tfstate" + -join ((48..57) + (97..102) | Get-Random -Count 8 | % {{[char]$_}})
$ContainerName = "tfstate"
$Location = "{AZURE_LOCATION}"

Write-Host "🔧 Setting up Terraform remote state backend..." -ForegroundColor Cyan

# Create resource group
az group create --name $ResourceGroup --location $Location

# Create storage account
az storage account create `
  --name $StorageAccount `
  --resource-group $ResourceGroup `
  --location $Location `
  --sku Standard_LRS `
  --encryption-services blob

# Get storage account key
$AccountKey = (az storage account keys list `
  --resource-group $ResourceGroup `
  --account-name $StorageAccount `
  --query '[0].value' -o tsv)

# Create blob container
az storage container create `
  --name $ContainerName `
  --account-name $StorageAccount `
  --account-key $AccountKey

Write-Host "✅ Remote state backend created!" -ForegroundColor Green
Write-Host ""
Write-Host "📋 Update backend.tf with these values:" -ForegroundColor Yellow
Write-Host "   resource_group_name  = `"$ResourceGroup`""
Write-Host "   storage_account_name = `"$StorageAccount`""
Write-Host "   container_name       = `"$ContainerName`""
Write-Host ""
Write-Host "🔐 Initialize Terraform with remote backend:" -ForegroundColor Cyan
Write-Host "   terraform init -backend-config=`"backend.tf`""
"""

with open(f"{azure_dir}/setup_backend.ps1", "w") as f:
    f.write(setup_ps1)

print("✅ Remote state backend configuration created:")
print(f"   - {azure_dir}/backend.tf.example")
print(f"   - {azure_dir}/setup_backend.sh")
print(f"   - {azure_dir}/setup_backend.ps1")
print("\n📋 To set up remote state:")
print("   1. Run: ./setup_backend.sh (or setup_backend.ps1 on Windows)")
print("   2. Copy values to backend.tf")
print("   3. Run: terraform init -migrate-state")
print("\n💡 Remote state benefits:")
print("   - Team collaboration (shared state)")
print("   - State locking (prevents conflicts)")
print("   - Encryption at rest")
print("   - Automatic backups")
print("   - Version history")
print("\n⚠️  Important:")
print("   - Never commit state files to Git")
print("   - Store backend config in secure location")
print("   - Use separate state files per environment")